In [1]:
import numpy as np
from subprocess import PIPE, run
import matplotlib.pyplot as plt
import os
import textwrap
from waxx.control import ethernet_relay

class ExptBuilder():
    def __init__(self):
        self.__code_path__ = os.environ.get('code')
        self.__temp_exp_path__ = os.path.join(self.__code_path__, "k-exp", "kexp", "experiments", "ml_expt.py")

    def run_expt(self):
        expt_path = self.__temp_exp_path__
        run_expt_command = r"%kpy% & artiq_run " + expt_path
        result = run(run_expt_command, stdout=PIPE, stderr=PIPE, universal_newlines=True, shell=True)
        print(result.returncode, result.stdout, result.stderr)
        os.remove(self.__temp_exp_path__)
        return result.returncode
    
    def write_experiment_to_file(self, program):
        with open(self.__temp_exp_path__, 'w') as file:
            file.write(program)
    
    def fringe_scan_expt(self, img_amp, freq_raman):
        script = textwrap.dedent(f"""
        from artiq.experiment import *
        from artiq.experiment import delay
        from kexp import Base, img_types, cameras
        import numpy as np
        from kexp.calibrations.tweezer import tweezer_vpd1_to_vpd2
        from kexp.calibrations.imaging import high_field_imaging_detuning
        from artiq.coredevice.sampler import Sampler
        from artiq.language import now_mu
        from kexp.util.artiq.async_print import aprint

        class hf_monitored_rabi(EnvExperiment, Base):

            def prepare(self):
                Base.__init__(self,setup_camera=True,
                            camera_select=cameras.andor,
                            save_data=True,
                            imaging_type=img_types.DISPERSIVE)

                
                self.p.frequency_raman_transition = {freq_raman}

                self.p.t_continuous_rabi = 450.e-6
                
                self.p.amp_imaging = {img_amp}

                self.p.hf_imaging_detuning = -568.e6 # 182.

                self.p.hf_imaging_detuning_mid = -514.e6 # -1 --> 0
                
                self.p.dimension_slm_mask = 20.e-6

                self.p.phase_slm_mask = 0.186 * np.pi

                self.p.t_tweezer_hold = 20.e-3

                self.p.t_tof = 20.e-6
                
                self.p.t_mot_load = 1.0
                
                self.p.N_repeats = 15

                self.scope = self.scope_data.add_siglent_scope("192.168.1.108", label='PD', arm=False)

                self.finish_prepare(shuffle=True)

            @kernel
            def scan_kernel(self):
                
                self.set_imaging_detuning(frequency_detuned = self.p.hf_imaging_detuning_mid)
                self.slm.write_phase_mask_kernel(phase=self.p.phase_slm_mask,dimension=self.p.dimension_slm_mask)
                self.imaging.set_power(self.p.amp_imaging)

                self.prepare_hf_tweezers(squeeze=True)

                self.raman.init(fraction_power = self.p.fraction_power_raman,
                                frequency_transition = self.p.frequency_raman_transition)

                self.ttl.raman_shutter.on()
                delay(10.e-3)
                self.ttl.line_trigger.wait_for_line_trigger()
                delay(4.7e-3)
                
                self.ttl.pd_scope_trig3.pulse(1.e-6)
                self.imaging.on()
                delay(3.e-6)

                self.raman.pulse(t=self.p.t_continuous_rabi)
                
                self.imaging.off()

                self.ttl.raman_shutter.off()
                
                self.set_imaging_detuning(frequency_detuned = self.p.hf_imaging_detuning)
                self.imaging.set_power(.2,reset_pid=True)

                delay(self.p.t_tweezer_hold)
                self.tweezer.off()

                delay(self.p.t_tof)

                self.abs_image()

                self.core.wait_until_mu(now_mu())
                self.scope.read_sweep(0)
                self.core.break_realtime()
                delay(30.e-3)

            @kernel
            def run(self):
                self.init_kernel()
                self.load_2D_mot(self.p.t_2D_mot_load_delay)
                self.scan()
                self.mot_observe()

            def analyze(self):
                import os
                expt_filepath = os.path.abspath(__file__)
                # aprint(self.scope._data)
                self.end(expt_filepath)
                """)
        return script

In [2]:
eBuilder = ExptBuilder()

In [3]:
img_amp = np.linspace(.2,2.5,100)

freq_raman = np.array([1.19495666e+08, 1.19499457e+08, 1.19503247e+08, 1.19507038e+08,
       1.19510828e+08, 1.19514619e+08, 1.19518410e+08, 1.19522200e+08,
       1.19525991e+08, 1.19529781e+08, 1.19533572e+08, 1.19537362e+08,
       1.19541153e+08, 1.19544944e+08, 1.19548734e+08, 1.19552525e+08,
       1.19556315e+08, 1.19560106e+08, 1.19563897e+08, 1.19567687e+08,
       1.19571478e+08, 1.19575268e+08, 1.19579059e+08, 1.19582849e+08,
       1.19586640e+08, 1.19590431e+08, 1.19594221e+08, 1.19598012e+08,
       1.19601802e+08, 1.19605593e+08, 1.19609384e+08, 1.19613174e+08,
       1.19616965e+08, 1.19620755e+08, 1.19624546e+08, 1.19628337e+08,
       1.19632127e+08, 1.19635918e+08, 1.19639708e+08, 1.19643499e+08,
       1.19647289e+08, 1.19651080e+08, 1.19654871e+08, 1.19658661e+08,
       1.19662452e+08, 1.19666242e+08, 1.19670033e+08, 1.19673824e+08,
       1.19677614e+08, 1.19681405e+08, 1.19685195e+08, 1.19688986e+08,
       1.19692776e+08, 1.19696567e+08, 1.19700358e+08, 1.19704148e+08,
       1.19707939e+08, 1.19711729e+08, 1.19715520e+08, 1.19719311e+08,
       1.19723101e+08, 1.19726892e+08, 1.19730682e+08, 1.19734473e+08,
       1.19738263e+08, 1.19742054e+08, 1.19745845e+08, 1.19749635e+08,
       1.19753426e+08, 1.19757216e+08, 1.19761007e+08, 1.19764798e+08,
       1.19768588e+08, 1.19772379e+08, 1.19776169e+08, 1.19779960e+08,
       1.19783751e+08, 1.19787541e+08, 1.19791332e+08, 1.19795122e+08,
       1.19798913e+08, 1.19802703e+08, 1.19806494e+08, 1.19810285e+08,
       1.19814075e+08, 1.19817866e+08, 1.19821656e+08, 1.19825447e+08,
       1.19829238e+08, 1.19833028e+08, 1.19836819e+08, 1.19840609e+08,
       1.19844400e+08, 1.19848190e+08, 1.19851981e+08, 1.19855772e+08,
       1.19859562e+08, 1.19863353e+08, 1.19867143e+08, 1.19870934e+08])


for f in range(len(img_amp)):
    print(img_amp[f],freq_raman[f])
    eBuilder.write_experiment_to_file(eBuilder.fringe_scan_expt(img_amp=img_amp[f],freq_raman = freq_raman[f]))
    eBuilder.run_expt()

0.2 119495666.0
0 Run id: 65188
 15 values of dummy. 15 total shots. 45 total images expected.
B:\_K\PotassiumData\2026-04-26\0065188_2026-04-26_14-37-11_hf_monitored_rabi.hdf5
Acknowledged camera ready signal.
Camera is ready.

Sent: {'mask': 'spot', 'center': [993, 820], 'phase': 0.186, 'dimension': 20, 'initialize': False, 'spacing': 10, 'angle': 45}
-> mask: spot, dimension = 20 um, phase = 0.186 pi, x-center = 993, y-center = 820

 Run ID: 65188

Sent: {'mask': 'spot', 'center': [993, 820], 'phase': 0.186, 'dimension': 20, 'initialize': False, 'spacing': 10, 'angle': 45}
-> mask: spot, dimension = 20 um, phase = 0.186 pi, x-center = 993, y-center = 820


Sent: {'mask': 'spot', 'center': [993, 820], 'phase': 0.186, 'dimension': 20, 'initialize': False, 'spacing': 10, 'angle': 45}
-> mask: spot, dimension = 20 um, phase = 0.186 pi, x-center = 993, y-center = 820


Sent: {'mask': 'spot', 'center': [993, 820], 'phase': 0.186, 'dimension': 20, 'initialize': False, 'spacing': 10, 'angle